# The Fractal Gap and the Hierarchy of Absence
## Executable notebook for seam decomposition, carry geometry, and local GF(2) structure in SHA-256

**Source manuscript:** *The Fractal Gap and the Hierarchy of Absence: A Geometric Inversion of Cryptographic Topology*  
**Author metadata in source:** Driven by Dean Kulik, April 2026  
**Notebook objective:** convert the manuscript into a reproducible, publication-style computational notebook suitable for external technical readers.

### Scope
This notebook does four things:

1. extracts and indexes the structure of the source document;
2. rebuilds the SHA-256 single-block compression engine and its round trace;
3. verifies the claims that are directly implied by explicit formulas or executable constructions in the manuscript;
4. labels the claims that require additional unpublished code or a more explicit mathematical specification than the document itself provides.

### Reproducibility standard
All numerical statements reported by this notebook are produced by executed code cells in this notebook.  
Whenever the manuscript introduces terminology such as *orbit*, *seam*, *gap*, or *universal seed*, the notebook maps those terms onto explicit binary operators, carry decompositions, local sensitivity matrices, and reverse recurrences.


In [1]:
from __future__ import annotations

import json
import re
import math
import random
import statistics
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from docx import Document

OUTPUT_DIR = Path("/mnt/data/fractal_gap_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RNG = random.Random(20260403)
MASK = 0xFFFFFFFF

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)


## 1. Source document ingestion and section map

The notebook begins by loading the source `.docx` and extracting a clean section outline.  
This provides context for external readers without adding commentary outside the manuscript's technical flow.


In [2]:
SOURCE_DOCX = Path("/mnt/data/Fractal Gap Hierarchy Confirmed (2).docx")
doc = Document(SOURCE_DOCX)

paragraphs = [p.text.strip() for p in doc.paragraphs]
paragraphs = [p for p in paragraphs if p]

section_map = []
current = {"title": "Front matter", "paragraphs": []}
for p in paragraphs:
    if re.match(r"^\d+(\.\d+)*\s", p):
        if current["paragraphs"] or current["title"] != "Front matter":
            section_map.append(current)
        current = {"title": p, "paragraphs": []}
    else:
        current["paragraphs"].append(p)
if current["paragraphs"] or current["title"] != "Front matter":
    section_map.append(current)

section_df = pd.DataFrame(
    {
        "section": [s["title"] for s in section_map],
        "paragraph_count": [len(s["paragraphs"]) for s in section_map],
        "first_sentence": [
            (s["paragraphs"][0][:140] + ("..." if len(s["paragraphs"][0]) > 140 else "")) if s["paragraphs"] else ""
            for s in section_map
        ],
    }
)

with open(OUTPUT_DIR / "source_section_map.json", "w", encoding="utf-8") as f:
    json.dump(section_map, f, indent=2)

section_df


,section,paragraph_count,first_sentence
0,Front matter,14,The Fractal Gap and the Hierarchy of Absence: ...
1,3.1 The 64-Cell Recurrence Architecture,7,The internal architecture of SHA-256 must be c...
2,3.2 Topological Comparison: Sequential Compres...,14,To fully grasp the unique vulnerability embedd...
3,6.1 Level 1: The Macro-View (The Orbit),3,"At the highest level of macro-observation, the..."
4,6.2 Level 2: The Operational View (The Seam),5,Descending into the intermediate computational...
5,6.3 Level 3: The Foundational View (The GF(2) ...,2,At the absolute deepest level of algorithmic a...
6,6.4 Synthesis of the Visual Mapping,4,The following table visually constructs the st...
7,7.1 The Rigid Architecture of the Glass Key,3,The 112-byte Glass Key is structurally partiti...
8,7.2 Harmonic Reconstruction and the Required O...,6,To forcefully reverse the topological collapse...
9,8.1 Cryptanalysis as Enterprise Array Reconstr...,4,To genuinely understand the magnitude of this ...


### Executable targets extracted from the manuscript

The manuscript contains a mix of narrative claims, explicit formulas, and forward references to additional code artifacts.  
For a professional computational notebook, the targets should be separated into:

- **directly reproducible**: formulas and constructions explicitly specified in the manuscript;
- **reproducible with operational interpretation**: claims that require a precise executable definition of terms such as *seam* or *orbit*;
- **not fully specified in the manuscript**: claims that reference external notebooks, unpublished scripts, or unnamed intermediate operators.

The table below states the executable program used in this notebook.


In [3]:
targets_df = pd.DataFrame(
    [
        {
            "target": "Round-0 ground witness",
            "status": "Directly reproducible",
            "notebook implementation": r"$T2_0 = \Sigma_0(a_0) + Maj(a_0,b_0,c_0)$ on the SHA-256 initial rails",
        },
        {
            "target": "NOP backbone",
            "status": "Directly reproducible",
            "notebook implementation": r"Run the 64-round compressor with $W_r = 0$ for all rounds",
        },
        {
            "target": "Seam decomposition",
            "status": "Reproducible with explicit operator choice",
            "notebook implementation": r"Use exact carry-save decomposition and the identity $x+y=(x\oplus y)+2(x\wedge y)$",
        },
        {
            "target": "33-36 bit local rank deficit",
            "status": "Reproducible with explicit local map",
            "notebook implementation": r"Construct a 192-bit active-state local GF(2) sensitivity matrix for $(a,b,c,e,f,g)\mapsto(a_1,b_1,c_1,e_1,f_1,g_1)$",
        },
        {
            "target": "Exact reverse walk of the message schedule",
            "status": "Directly reproducible",
            "notebook implementation": r"Recover $W_{0..15}$ from $W_{16..63}$ and the full schedule by backward substitution",
        },
        {
            "target": "Glass Key / 112-byte compression",
            "status": "Not fully specified in manuscript",
            "notebook implementation": "Not claimed as reproduced here without the external notebook or precise construction",
        },
        {
            "target": "Digest-only preimage recovery",
            "status": "Not fully specified in manuscript",
            "notebook implementation": "Not claimed as reproduced here because the manuscript does not provide a complete executable inversion protocol",
        },
    ]
)
targets_df


,target,status,notebook implementation
0,Round-0 ground witness,Directly reproducible,"$T2_0 = \Sigma_0(a_0) + Maj(a_0,b_0,c_0)$ on t..."
1,NOP backbone,Directly reproducible,Run the 64-round compressor with $W_r = 0$ for...
2,Seam decomposition,Reproducible with explicit operator choice,Use exact carry-save decomposition and the ide...
3,33-36 bit local rank deficit,Reproducible with explicit local map,Construct a 192-bit active-state local GF(2) s...
4,Exact reverse walk of the message schedule,Directly reproducible,Recover $W_{0..15}$ from $W_{16..63}$ and the ...
5,Glass Key / 112-byte compression,Not fully specified in manuscript,Not claimed as reproduced here without the ext...
6,Digest-only preimage recovery,Not fully specified in manuscript,Not claimed as reproduced here because the man...


## 2. SHA-256 reference engine

This section defines a complete single-block SHA-256 compression implementation with:

- exact round constants and initial rails;
- message padding and schedule expansion;
- full round trace capture;
- carry instrumentation for the two nonlinear injection channels \(T1\) and \(T2\).

The implementation is intentionally explicit rather than minimal so that every subsequent section can reuse the same trace object.


In [4]:
K = [
    0x428A2F98, 0x71374491, 0xB5C0FBCF, 0xE9B5DBA5, 0x3956C25B, 0x59F111F1, 0x923F82A4, 0xAB1C5ED5,
    0xD807AA98, 0x12835B01, 0x243185BE, 0x550C7DC3, 0x72BE5D74, 0x80DEB1FE, 0x9BDC06A7, 0xC19BF174,
    0xE49B69C1, 0xEFBE4786, 0x0FC19DC6, 0x240CA1CC, 0x2DE92C6F, 0x4A7484AA, 0x5CB0A9DC, 0x76F988DA,
    0x983E5152, 0xA831C66D, 0xB00327C8, 0xBF597FC7, 0xC6E00BF3, 0xD5A79147, 0x06CA6351, 0x14292967,
    0x27B70A85, 0x2E1B2138, 0x4D2C6DFC, 0x53380D13, 0x650A7354, 0x766A0ABB, 0x81C2C92E, 0x92722C85,
    0xA2BFE8A1, 0xA81A664B, 0xC24B8B70, 0xC76C51A3, 0xD192E819, 0xD6990624, 0xF40E3585, 0x106AA070,
    0x19A4C116, 0x1E376C08, 0x2748774C, 0x34B0BCB5, 0x391C0CB3, 0x4ED8AA4A, 0x5B9CCA4F, 0x682E6FF3,
    0x748F82EE, 0x78A5636F, 0x84C87814, 0x8CC70208, 0x90BEFFFA, 0xA4506CEB, 0xBEF9A3F7, 0xC67178F2,
]

H0 = [
    0x6A09E667, 0xBB67AE85, 0x3C6EF372, 0xA54FF53A,
    0x510E527F, 0x9B05688C, 0x1F83D9AB, 0x5BE0CD19,
]

def add32(*xs: int) -> int:
    return sum(xs) & MASK

def rotr(x: int, n: int) -> int:
    return ((x >> n) | ((x << (32 - n)) & MASK)) & MASK

def shr(x: int, n: int) -> int:
    return (x >> n) & MASK

def Ch(x: int, y: int, z: int) -> int:
    return ((x & y) ^ ((~x) & z)) & MASK

def Maj(x: int, y: int, z: int) -> int:
    return ((x & y) ^ (x & z) ^ (y & z)) & MASK

def Sigma0(x: int) -> int:
    return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)

def Sigma1(x: int) -> int:
    return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)

def sigma0(x: int) -> int:
    return rotr(x, 7) ^ rotr(x, 18) ^ shr(x, 3)

def sigma1(x: int) -> int:
    return rotr(x, 17) ^ rotr(x, 19) ^ shr(x, 10)

def pad_one_block(message: bytes) -> bytes:
    if len(message) > 55:
        raise ValueError("This notebook focuses on the single-block case (message length <= 55 bytes).")
    bit_len = len(message) * 8
    padded = message + b"\x80"
    while (len(padded) % 64) != 56:
        padded += b"\x00"
    padded += bit_len.to_bytes(8, byteorder="big")
    assert len(padded) == 64
    return padded

def block_words(block: bytes) -> list[int]:
    return [int.from_bytes(block[i:i+4], "big") for i in range(0, 64, 4)]

def build_schedule(words16: list[int]) -> list[int]:
    W = list(words16)
    for t in range(16, 64):
        W.append(add32(sigma1(W[t - 2]), W[t - 7], sigma0(W[t - 15]), W[t - 16]))
    return W

def reverse_schedule(W: list[int]) -> list[int]:
    W = list(W)
    if len(W) != 64:
        raise ValueError("Expected a 64-word schedule.")
    for t in range(63, 15, -1):
        recovered = (W[t] - sigma1(W[t - 2]) - W[t - 7] - sigma0(W[t - 15])) & MASK
        assert recovered == W[t - 16]
    recovered_front = list(W[:16])
    return recovered_front

@dataclass
class RoundRecord:
    round_index: int
    a: int; b: int; c: int; d: int; e: int; f: int; g: int; h: int
    W: int; K: int
    sigma0_a: int; sigma1_e: int
    maj_abc: int; ch_efg: int
    T1: int; T2: int
    a1: int; e1: int

def compress_with_trace(schedule: list[int], initial_state: list[int] | None = None) -> tuple[list[int], list[RoundRecord]]:
    state = list(H0 if initial_state is None else initial_state)
    trace: list[RoundRecord] = []
    for r in range(64):
        a, b, c, d, e, f, g, h = state
        s0 = Sigma0(a)
        s1 = Sigma1(e)
        maj = Maj(a, b, c)
        ch = Ch(e, f, g)
        T1 = add32(h, s1, ch, K[r], schedule[r])
        T2 = add32(s0, maj)
        a1 = add32(T1, T2)
        e1 = add32(d, T1)
        trace.append(
            RoundRecord(
                round_index=r, a=a, b=b, c=c, d=d, e=e, f=f, g=g, h=h,
                W=schedule[r], K=K[r],
                sigma0_a=s0, sigma1_e=s1, maj_abc=maj, ch_efg=ch,
                T1=T1, T2=T2, a1=a1, e1=e1
            )
        )
        state = [a1, a, b, c, e1, e, f, g]
    final_state = [add32(x, y) for x, y in zip(H0, state)]
    return final_state, trace

def digest_hex_from_state(state: list[int]) -> str:
    return "".join(f"{x:08x}" for x in state)

def sha256_single_block(message: bytes) -> tuple[str, list[int], list[int], list[RoundRecord]]:
    block = pad_one_block(message)
    words16 = block_words(block)
    schedule = build_schedule(words16)
    final_state, trace = compress_with_trace(schedule)
    return digest_hex_from_state(final_state), words16, schedule, trace

def words_to_bits(words: list[int]) -> np.ndarray:
    bits = []
    for w in words:
        bits.extend([(w >> i) & 1 for i in range(32)])
    return np.array(bits, dtype=np.uint8)

def popcount32(x: int) -> int:
    return int(x & MASK).bit_count()


## 3. Exact round-0 invariants and the NOP backbone

The manuscript states that the round-0 ground witness is fixed by the initial rails and that the message enters the compressor as a displacement on top of that ground plane.

This is directly testable.  
The notebook verifies:

\[
T2_0 = \Sigma_0(a_0) + Maj(a_0,b_0,c_0) = 0x08909ae5,
\]

and constructs the 64-round *NOP backbone* by setting \(W_r=0\) for every round.


In [5]:
# Round-0 ground witness
a0, b0, c0, d0, e0, f0, g0, h0 = H0
T2_0 = add32(Sigma0(a0), Maj(a0, b0, c0))

# NOP backbone
nop_schedule = [0] * 64
nop_final_state, nop_trace = compress_with_trace(nop_schedule)

ground_summary = {
    "T2_0_hex": f"0x{T2_0:08x}",
    "T2_0_decimal": T2_0,
    "nop_digest": digest_hex_from_state(nop_final_state),
    "nop_first_round_T1": f"0x{nop_trace[0].T1:08x}",
    "nop_first_round_T2": f"0x{nop_trace[0].T2:08x}",
}

with open(OUTPUT_DIR / "ground_summary.json", "w", encoding="utf-8") as f:
    json.dump(ground_summary, f, indent=2)

pd.DataFrame([ground_summary])


,T2_0_hex,T2_0_decimal,nop_digest,nop_first_round_T1,nop_first_round_T2
0,0x08909ae5,143694565,da5698be17b9b46962335799779fbeca8ce5d491c0d262...,0xf377ed68,0x08909ae5


### Perturbation identity at round 0

For the first round, with the rails fixed, the manuscript claims that the message displacement enters through \(W_0\) as a direct perturbation of \(T1_0\).  
This can be checked on any one-block message.


In [6]:
example_message = b"abc"
example_digest, example_words16, example_schedule, example_trace = sha256_single_block(example_message)

round0_identity_df = pd.DataFrame(
    [
        {
            "message": example_message.decode("ascii"),
            "W0_hex": f"0x{example_trace[0].W:08x}",
            "T1_signal_hex": f"0x{example_trace[0].T1:08x}",
            "T1_nop_hex": f"0x{nop_trace[0].T1:08x}",
            "difference_hex": f"0x{((example_trace[0].T1 - nop_trace[0].T1) & MASK):08x}",
            "matches_W0": ((example_trace[0].T1 - nop_trace[0].T1) & MASK) == example_trace[0].W,
        }
    ]
)

round0_identity_df


,message,W0_hex,T1_signal_hex,T1_nop_hex,difference_hex,matches_W0
0,abc,0x61626380,0x54da50e8,0xf377ed68,0x61626380,True


### Example trace for a standard test message

The trace below gives the first several rounds for the standard single-block message `b"abc"`.  
This is the reference state evolution used in later sections.


In [7]:
trace_df = pd.DataFrame([asdict(r) for r in example_trace])
trace_df.head(8)


,round_index,a,b,c,d,e,f,g,h,W,K,sigma0_a,sigma1_e,maj_abc,ch_efg,T1,T2,a1,e1
0,0,1779033703,3144134277,1013904242,2773480762,1359893119,2600822924,528734635,1541459225,1633837952,1116352408,3458249854,898049835,980412007,528861580,1423593704,143694565,1567288269,4197074466
1,1,1567288269,1779033703,3144134277,1013904242,4197074466,1359893119,2600822924,528734635,0,1899447441,2728355025,1519713581,2070671045,1359964846,1012893207,504058774,1516951981,2026797449
2,2,1516951981,1567288269,1779033703,3144134277,2026797449,4197074466,1359893119,2600822924,0,3049323471,815190100,1945166997,1516956653,2030715510,1036094310,2332146753,3368241063,4180228587
3,3,3368241063,1516951981,1567288269,1779033703,4180228587,2026797449,4197074466,1359893119,0,3921009573,3255830736,101151804,1483393965,2047508361,3134595561,444257405,3578852966,618661968
4,4,3578852966,3368241063,1516951981,1567288269,618661968,4180228587,2026797449,4197074466,0,961987163,1169886283,982725682,3628259239,2016311753,3863131768,503178226,71342698,1135452741
5,5,71342698,3578852966,3368241063,1516951981,1135452741,618661968,4180228587,2026797449,0,1508970993,1344908431,2339889564,3292583526,3097500138,383223552,342524661,725748213,1900175533
6,6,725748213,71342698,3578852966,3368241063,1900175533,1135452741,618661968,4180228587,0,2453635748,224267456,22771574,88119910,1168123989,3529792602,312387366,3842179968,2603066369
7,7,3842179968,725748213,71342698,3578852966,2603066369,1900175533,1135452741,618661968,0,2870763221,692710252,361626281,625085408,1368007237,924091411,1317795660,2241887071,207977081


## 4. Exact seam decomposition via carry-save arithmetic

The manuscript's *universal seed* language can be made precise using standard carry-save arithmetic.

For two operands:
\[
x + y = (x \oplus y) + 2(x \wedge y) \pmod{2^{32}}.
\]

For three operands:
\[
x + y + z = s + c \pmod{2^{32}},
\]
where
\[
s = x \oplus y \oplus z,\qquad
c = 2\big((x\wedge y)\vee(x\wedge z)\vee(y\wedge z)\big).
\]

This section builds an exact carry-save decomposition for both nonlinear injection channels:

- \(T2 = \Sigma_0(a) + Maj(a,b,c)\);
- \(T1 = h + \Sigma_1(e) + Ch(e,f,g) + K + W\).

The seam is therefore represented by a linear-looking XOR layer together with a carry-residual layer that is tracked explicitly instead of discarded.


In [8]:
def carry_seed_pair(x: int, y: int) -> int:
    return (x & y) & MASK

def seam_pair(x: int, y: int) -> tuple[int, int]:
    xor_part = (x ^ y) & MASK
    carry_part = ((x & y) << 1) & MASK
    return xor_part, carry_part

def csa3(x: int, y: int, z: int) -> tuple[int, int]:
    s = (x ^ y ^ z) & MASK
    c = (((x & y) | (x & z) | (y & z)) << 1) & MASK
    return s, c

def decompose_T2(rec: RoundRecord) -> dict:
    xor_part, carry_part = seam_pair(rec.sigma0_a, rec.maj_abc)
    assert add32(xor_part, carry_part) == rec.T2
    return {
        "xor_part": xor_part,
        "carry_part": carry_part,
        "carry_seed": carry_seed_pair(rec.sigma0_a, rec.maj_abc),
    }

def decompose_T1(rec: RoundRecord) -> dict:
    x1, x2, x3, x4, x5 = rec.h, rec.sigma1_e, rec.ch_efg, rec.K, rec.W
    s1, c1 = csa3(x1, x2, x3)
    s2, c2 = csa3(s1, x4, x5)
    final_xor, final_carry = seam_pair(c1, c2)
    reconstructed = add32(s2, final_xor, final_carry)
    assert reconstructed == rec.T1
    return {
        "stage1_sum": s1,
        "stage1_carry": c1,
        "stage2_sum": s2,
        "stage2_carry": c2,
        "carry_merge_xor": final_xor,
        "carry_merge_carry": final_carry,
    }

seam_rows = []
for rec in example_trace:
    d2 = decompose_T2(rec)
    d1 = decompose_T1(rec)
    seam_rows.append(
        {
            "round": rec.round_index,
            "T2_xor_weight": popcount32(d2["xor_part"]),
            "T2_carry_weight": popcount32(d2["carry_part"]),
            "T1_stage1_carry_weight": popcount32(d1["stage1_carry"]),
            "T1_stage2_carry_weight": popcount32(d1["stage2_carry"]),
            "T1_carry_merge_weight": popcount32(d1["carry_merge_carry"]),
            "T1_final_sum_weight": popcount32(rec.T1),
        }
    )

seam_df = pd.DataFrame(seam_rows)
seam_df.head(10)


,round,T2_xor_weight,T2_carry_weight,T1_stage1_carry_weight,T1_stage2_carry_weight,T1_carry_merge_weight,T1_final_sum_weight
0,0,16,10,15,13,4,14
1,1,15,11,21,4,2,17
2,2,20,6,11,13,4,16
3,3,19,3,16,9,2,17
4,4,22,5,14,11,4,16
5,5,17,5,15,11,3,13
6,6,13,5,15,5,0,15
7,7,11,8,10,7,0,12
8,8,18,5,14,6,3,13
9,9,17,7,12,6,0,16


### Carry-residual profile across the 64 rounds

The manuscript frames the non-linear layer as a hierarchy of residuals rather than lost information.  
The figure below shows the carry-bearing components of the exact seam decomposition across all 64 rounds for the example message.


In [9]:
fig = go.Figure()
fig.add_trace(go.Bar(name="T2 carry weight", x=seam_df["round"], y=seam_df["T2_carry_weight"]))
fig.add_trace(go.Bar(name="T1 stage-1 carry weight", x=seam_df["round"], y=seam_df["T1_stage1_carry_weight"]))
fig.add_trace(go.Bar(name="T1 stage-2 carry weight", x=seam_df["round"], y=seam_df["T1_stage2_carry_weight"]))
fig.add_trace(go.Bar(name="T1 carry-merge weight", x=seam_df["round"], y=seam_df["T1_carry_merge_weight"]))
fig.update_layout(
    barmode="group",
    title="Carry-bearing components in the exact seam decomposition",
    xaxis_title="Round",
    yaxis_title="Bit weight",
    template="plotly_white",
    legend_title_text="Component",
)
fig.show()


## 5. Orbit-level carry signatures

At the coarse scale, the manuscript describes an aggregate carry load over the 64-round execution.  
A direct executable surrogate for that quantity is the per-round carry bit of \(T2\), together with the multi-term carry profile extracted from the \(T1\) decomposition.

This section compares the NOP backbone against random single-block messages by measuring Hamming distance between 64-round \(T2\) carry signatures.


In [10]:
def t2_carry_signature(trace: list[RoundRecord]) -> int:
    signature = 0
    for rec in trace:
        carry = 1 if (rec.sigma0_a + rec.maj_abc) >> 32 else 0
        signature = (signature << 1) | carry
    return signature

def t2_carry_bitlist(trace: list[RoundRecord]) -> list[int]:
    return [1 if (rec.sigma0_a + rec.maj_abc) >> 32 else 0 for rec in trace]

nop_signature = t2_carry_signature(nop_trace)
nop_bits = t2_carry_bitlist(nop_trace)

samples = []
for i in range(200):
    msg_len = RNG.randint(0, 55)
    msg = bytes(RNG.getrandbits(8) for _ in range(msg_len))
    _, _, _, tr = sha256_single_block(msg)
    sig = t2_carry_signature(tr)
    dist = (sig ^ nop_signature).bit_count()
    samples.append({"sample_index": i, "message_length": msg_len, "hamming_distance_to_nop": dist})

carry_dist_df = pd.DataFrame(samples)
carry_dist_summary = {
    "nop_signature_hex": f"0x{nop_signature:016x}",
    "mean_hamming_distance": float(carry_dist_df["hamming_distance_to_nop"].mean()),
    "std_hamming_distance": float(carry_dist_df["hamming_distance_to_nop"].std(ddof=1)),
    "min_hamming_distance": int(carry_dist_df["hamming_distance_to_nop"].min()),
    "max_hamming_distance": int(carry_dist_df["hamming_distance_to_nop"].max()),
}

with open(OUTPUT_DIR / "carry_signature_summary.json", "w", encoding="utf-8") as f:
    json.dump(carry_dist_summary, f, indent=2)

pd.DataFrame([carry_dist_summary])


,nop_signature_hex,mean_hamming_distance,std_hamming_distance,min_hamming_distance,max_hamming_distance
0,0xde1a54568d60f7b6,31.55,4.235232,20,42


In [11]:
fig = px.histogram(
    carry_dist_df,
    x="hamming_distance_to_nop",
    nbins=20,
    title="Distribution of T2 carry-signature distance from the NOP backbone (200 random messages)",
    template="plotly_white",
)
fig.update_xaxes(title="Hamming distance to NOP 64-bit T2 carry signature")
fig.update_yaxes(title="Count")
fig.show()


## 6. Local GF(2) Jacobian study on a 192-bit active-state operator

The manuscript reports a corrected local rank deficit in a 192-dimensional setting.  
A precise executable realization of that setting is the following **active-state round map**:

\[
F_r : (a,b,c,e,f,g) \mapsto (a_{r+1}, b_{r+1}, c_{r+1}, e_{r+1}, f_{r+1}, g_{r+1}),
\]

with \(d_r\), \(h_r\), \(W_r\), and \(K_r\) held fixed at the round state.

This is a \(192 \to 192\) map.  
Its local GF(2) sensitivity matrix is constructed by toggling each input bit once and recording the output-bit difference vector. This is not a symbolic derivative in the algebraic-geometry sense; it is a local binary sensitivity operator. It is, however, explicit, reproducible, and directly tied to the manuscript's active-state language.


In [12]:
def gf2_rank(M: np.ndarray) -> int:
    A = M.copy()
    m, n = A.shape
    rank = 0
    for col in range(n):
        pivot = None
        for r in range(rank, m):
            if A[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != rank:
            A[[rank, pivot]] = A[[pivot, rank]]
        rows = np.where(A[:, col] == 1)[0]
        rows = rows[rows != rank]
        A[rows] ^= A[rank]
        rank += 1
        if rank == m:
            break
    return rank

def gf2_nullspace(A: np.ndarray) -> np.ndarray:
    R = A.copy()
    m, n = R.shape
    pivots = []
    row = 0
    for col in range(n):
        pivot = None
        for r in range(row, m):
            if R[r, col]:
                pivot = r
                break
        if pivot is None:
            continue
        if pivot != row:
            R[[row, pivot]] = R[[pivot, row]]
        rows = np.where(R[:, col] == 1)[0]
        rows = rows[rows != row]
        R[rows] ^= R[row]
        pivots.append(col)
        row += 1
        if row == m:
            break
    free = [c for c in range(n) if c not in pivots]
    basis = []
    for f in free:
        x = np.zeros(n, dtype=np.uint8)
        x[f] = 1
        for i, p in enumerate(pivots):
            s = 0
            for j in free:
                if R[i, j] and x[j]:
                    s ^= 1
            x[p] = s
        basis.append(x)
    return np.stack(basis) if basis else np.zeros((0, n), dtype=np.uint8)

def active6_map(a: int, b: int, c: int, d: int, e: int, f: int, g: int, h: int, w: int, k: int) -> list[int]:
    T1 = add32(h, Sigma1(e), Ch(e, f, g), k, w)
    T2 = add32(Sigma0(a), Maj(a, b, c))
    a1 = add32(T1, T2)
    e1 = add32(d, T1)
    return [a1, a, b, e1, e, f]

def active6_local_matrix(rec: RoundRecord) -> np.ndarray:
    base_in = [rec.a, rec.b, rec.c, rec.e, rec.f, rec.g]
    base_out = words_to_bits(active6_map(rec.a, rec.b, rec.c, rec.d, rec.e, rec.f, rec.g, rec.h, rec.W, rec.K))
    cols = []
    for word_idx in range(6):
        for bit_idx in range(32):
            trial = base_in.copy()
            trial[word_idx] ^= (1 << bit_idx)
            out = active6_map(trial[0], trial[1], trial[2], rec.d, trial[3], trial[4], trial[5], rec.h, rec.W, rec.K)
            cols.append(base_out ^ words_to_bits(out))
    return np.stack(cols, axis=1)

rank_rows = []
for rec in nop_trace:
    J = active6_local_matrix(rec)
    rank = gf2_rank(J)
    rank_rows.append(
        {
            "round": rec.round_index,
            "rank": rank,
            "nullity": 192 - rank,
        }
    )

rank_df = pd.DataFrame(rank_rows)
rank_df.head()


,round,rank,nullity
0,0,159,33
1,1,158,34
2,2,161,31
3,3,153,39
4,4,166,26


### Rank/nullity profile on the NOP backbone

This is the central executable result of the notebook's local-operator study.


In [13]:
rank_summary = {
    "round0_rank": int(rank_df.loc[rank_df["round"] == 0, "rank"].iloc[0]),
    "round0_nullity": int(rank_df.loc[rank_df["round"] == 0, "nullity"].iloc[0]),
    "min_rank_over_nop_backbone": int(rank_df["rank"].min()),
    "max_rank_over_nop_backbone": int(rank_df["rank"].max()),
    "min_nullity_over_nop_backbone": int(rank_df["nullity"].min()),
    "max_nullity_over_nop_backbone": int(rank_df["nullity"].max()),
    "mean_rank_over_nop_backbone": float(rank_df["rank"].mean()),
    "mean_nullity_over_nop_backbone": float(rank_df["nullity"].mean()),
}

rank_df.to_csv(OUTPUT_DIR / "active6_rank_by_round.csv", index=False)
with open(OUTPUT_DIR / "active6_rank_summary.json", "w", encoding="utf-8") as f:
    json.dump(rank_summary, f, indent=2)

pd.DataFrame([rank_summary])


,round0_rank,round0_nullity,min_rank_over_nop_backbone,max_rank_over_nop_backbone,min_nullity_over_nop_backbone,max_nullity_over_nop_backbone,mean_rank_over_nop_backbone,mean_nullity_over_nop_backbone
0,159,33,153,167,25,39,159.859375,32.140625


In [14]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=rank_df["round"], y=rank_df["rank"], mode="lines+markers", name="Rank"))
fig.add_trace(go.Scatter(x=rank_df["round"], y=rank_df["nullity"], mode="lines+markers", name="Nullity", yaxis="y2"))
fig.update_layout(
    title="Local GF(2) sensitivity rank for the 192-bit active-state round operator on the NOP backbone",
    template="plotly_white",
    xaxis_title="Round",
    yaxis=dict(title="Rank"),
    yaxis2=dict(title="Nullity", overlaying="y", side="right"),
    legend=dict(x=0.01, y=0.99),
)
fig.show()


**Interpretation.**  
For this explicitly defined \(192 \to 192\) local operator:

- round 0 has **rank 159** and **nullity 33**;
- the full NOP backbone exhibits a rank band from **153 to 167** and a nullity band from **25 to 39**.

That means the manuscript's reported *33-36 bit deficit* is realized directly for round 0 and occurs as part of the observed NOP-backbone band under this local operator definition.  
The notebook therefore replaces an underspecified narrative claim with a concrete, reproducible sensitivity model.


### Null-space basis at round 0

To characterize the local blind directions explicitly, the notebook computes a GF(2) null-space basis for the round-0 active-state matrix and visualizes the input-bit support of the resulting basis vectors.


In [15]:
J0 = active6_local_matrix(nop_trace[0])
null_basis = gf2_nullspace(J0)

basis_support = []
for idx, vec in enumerate(null_basis):
    row = {"basis_vector": idx}
    for word_idx, word_name in enumerate(["a", "b", "c", "e", "f", "g"]):
        row[word_name] = int(vec[word_idx * 32 : (word_idx + 1) * 32].sum())
    basis_support.append(row)

basis_support_df = pd.DataFrame(basis_support)
basis_support_df.head()


,basis_vector,a,b,c,e,f,g
0,0,0,0,1,0,0,0
1,1,0,0,1,0,0,0
2,2,0,0,1,0,0,0
3,3,0,0,1,0,0,0
4,4,0,0,1,0,0,0


In [16]:
heatmap_values = basis_support_df[["a", "b", "c", "e", "f", "g"]].to_numpy()
fig = px.imshow(
    heatmap_values,
    labels=dict(x="Input word", y="Null-space basis vector index", color="Support weight"),
    x=["a", "b", "c", "e", "f", "g"],
    y=[str(i) for i in basis_support_df["basis_vector"]],
    title="Round-0 active-state null-space basis support by input word",
    aspect="auto",
    color_continuous_scale="Viridis",
)
fig.update_layout(template="plotly_white")
fig.show()


## 7. Rotation-operator audit

The manuscript points to the SHA-256 rotation sets

- \(\{2,13,22\}\) for \(\Sigma_0\),
- \(\{6,11,25\}\) for \(\Sigma_1\),

as structural boundaries in the local geometry.

A direct executable audit begins with the linear operators themselves.  
The notebook constructs the \(32\times32\) GF(2) matrices for \(\Sigma_0\) and \(\Sigma_1\) and reports their ranks.


In [17]:
def rotr_matrix(n: int, r: int) -> np.ndarray:
    M = np.zeros((n, n), dtype=np.uint8)
    for i in range(n):
        M[i, (i + r) % n] = 1
    return M

def sigma_matrix(rotations: tuple[int, int, int]) -> np.ndarray:
    M = np.zeros((32, 32), dtype=np.uint8)
    for r in rotations:
        M ^= rotr_matrix(32, r)
    return M

Sigma0_mat = sigma_matrix((2, 13, 22))
Sigma1_mat = sigma_matrix((6, 11, 25))

rotation_audit_df = pd.DataFrame(
    [
        {"operator": "Sigma0", "rotations": "{2,13,22}", "rank": gf2_rank(Sigma0_mat), "nullity": 32 - gf2_rank(Sigma0_mat)},
        {"operator": "Sigma1", "rotations": "{6,11,25}", "rank": gf2_rank(Sigma1_mat), "nullity": 32 - gf2_rank(Sigma1_mat)},
    ]
)
rotation_audit_df


,operator,rotations,rank,nullity
0,Sigma0,"{2,13,22}",32,0
1,Sigma1,"{6,11,25}",32,0


For the standalone \(32\)-bit linear rotation operators, the rank is full:

\[
\mathrm{rank}(\Sigma_0) = \mathrm{rank}(\Sigma_1) = 32.
\]

So any local null space observed in the active-state operator is **not** coming from the rotation layers alone.  
It arises from the interaction among:

- rotations,
- \(Ch\),
- \(Maj\),
- modular carry propagation,
- and the shift-injection recurrence.

That is exactly the regime where the local 192-bit operator is informative.


## 8. Exact reverse walk of the message schedule

The manuscript emphasizes deterministic schedule recovery.  
The precise executable statement supported by the specification is:

> if the full 64-word schedule is available, then the front 16 words can be recovered exactly by backward substitution.

This is a strong and useful property, and it is fully reproducible.


In [18]:
recovered_front = reverse_schedule(example_schedule)
schedule_reverse_df = pd.DataFrame(
    {
        "index": list(range(16)),
        "original_Wt": [f"0x{x:08x}" for x in example_schedule[:16]],
        "recovered_Wt": [f"0x{x:08x}" for x in recovered_front],
        "match": [x == y for x, y in zip(example_schedule[:16], recovered_front)],
    }
)

with open(OUTPUT_DIR / "schedule_reverse_validation.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "message": example_message.decode("ascii"),
            "all_match": bool(all(schedule_reverse_df["match"])),
            "front_words": recovered_front,
        },
        f,
        indent=2,
    )

schedule_reverse_df


,index,original_Wt,recovered_Wt,match
0,0,0x61626380,0x61626380,True
1,1,0x00000000,0x00000000,True
2,2,0x00000000,0x00000000,True
3,3,0x00000000,0x00000000,True
4,4,0x00000000,0x00000000,True
5,5,0x00000000,0x00000000,True
6,6,0x00000000,0x00000000,True
7,7,0x00000000,0x00000000,True
8,8,0x00000000,0x00000000,True
9,9,0x00000000,0x00000000,True


## 9. Reproducibility summary

This notebook converts the manuscript into a complete executable research artifact with the following verified results.

1. **Ground witness**
   - \(T2_0 = 0x08909ae5\) on the standard SHA-256 initial rails.

2. **NOP backbone**
   - The 64-round zero-schedule backbone is constructed explicitly and used as a stable reference trace.

3. **Round-0 perturbation**
   - For the standard test message `abc`, the round-0 perturbation identity
     \[
     T1_0 - T1_0^{(NOP)} \equiv W_0 \pmod{2^{32}}
     \]
     is verified exactly.

4. **Exact seam decomposition**
   - Both nonlinear injection channels are decomposed into XOR-like and carry-bearing layers using exact carry-save arithmetic.

5. **Local 192-bit sensitivity operator**
   - A precise active-state \(192 \to 192\) local GF(2) operator is defined and measured.
   - Round 0 yields rank \(159\), nullity \(33\).
   - The NOP backbone spans a rank band of \(153\) to \(167\).

6. **Rotation-layer audit**
   - \(\Sigma_0\) and \(\Sigma_1\) are full-rank as standalone GF(2) operators, so the local null space is a joint property of the full round mechanism rather than a trivial property of the rotation matrices alone.

7. **Schedule reverse walk**
   - The first 16 schedule words are recovered exactly from the full expanded schedule by backward substitution.

### Claims intentionally not asserted as reproduced here
The manuscript references additional artifacts and stronger inversion claims that are not fully specified in the `.docx` itself.  
This notebook therefore does **not** claim to reproduce, prove, or validate:

- digest-only preimage recovery;
- the full Glass Key compression protocol;
- the 112-byte packaging claim;
- any inversion result that depends on an external notebook not included in this manuscript.

That boundary is part of the notebook's scientific discipline: it reports what is executed here and separates it from what would require additional source material.


In [19]:
summary_table = pd.DataFrame(
    [
        {"result": "T2_0", "value": f"0x{T2_0:08x}"},
        {"result": "Round-0 active-state rank", "value": int(rank_df.loc[rank_df['round'] == 0, 'rank'].iloc[0])},
        {"result": "Round-0 active-state nullity", "value": int(rank_df.loc[rank_df['round'] == 0, 'nullity'].iloc[0])},
        {"result": "NOP rank range", "value": f"{rank_df['rank'].min()} to {rank_df['rank'].max()}"},
        {"result": "NOP nullity range", "value": f"{rank_df['nullity'].min()} to {rank_df['nullity'].max()}"},
        {"result": "T2 carry-signature mean distance to NOP", "value": round(float(carry_dist_df['hamming_distance_to_nop'].mean()), 3)},
        {"result": "Schedule reverse walk exact", "value": bool(all(schedule_reverse_df['match']))},
        {"result": "Example digest (abc)", "value": example_digest},
    ]
)
summary_table


,result,value
0,T2_0,0x08909ae5
1,Round-0 active-state rank,159
2,Round-0 active-state nullity,33
3,NOP rank range,153 to 167
4,NOP nullity range,25 to 39
5,T2 carry-signature mean distance to NOP,31.55
6,Schedule reverse walk exact,True
7,Example digest (abc),ba7816bf8f01cfea414140de5dae2223b00361a396177a...
